In [3]:
import pandas as pd

df = pd.read_csv(r'C:\Users\moham\OneDrive\Desktop\WebScraping\legacy_app\output\baby_banana\baby_banana.csv')
df.columns

Index(['Product', 'Handle', 'Price', 'SKU', 'Availability', 'Variant Title',
       'Variant ID', 'Weight', 'Inventory Management', 'Product URL',
       'Scraped at', 'Description'],
      dtype='object')

In [9]:
import pandas as pd

# 1. Clean the 'Description' column BEFORE merging to remove new lines
df['Description'] = df['Description'].str.replace(r'\n|\r', ' ', regex=True)

# 2. Updated aggregation rules to create clean strings instead of Python lists
# We use ', '.join() to make "White, Cream" instead of "['White', 'Cream']"
aggregation_rules = {
    # 'Title': lambda x: ', '.join(map(str, x.unique())),
    # 'Variant': lambda x: ', '.join(map(str, x.unique())), 
    'Variant ID': lambda x: ', '.join(map(str, x.unique())),
    'SKU': lambda x: ', '.join(map(str, x.unique())),
    # 'Size': lambda x: ', '.join(map(str, x.unique())),
    'Inventory Management': 'first',
    'Handle': 'first',
    'Availability': 'first',
    'Product URL': 'first',
    'Scraped at': 'first'
}

# Group by the "Core" identity
df_merged = df.groupby(['Product', 'Description', 'Price'], as_index=False).agg(aggregation_rules)

# 3. FIX: Create the Hollistic Description correctly
# Use .astype(str) instead of str() to handle it row-by-row
df_merged['Hollistic Description'] = (
    df_merged['Product'] + 
    # ' - ' + 
    # df_merged['Variant'].astype(str) + 
    ": " + 
    df_merged["Description"]
)

# Reorder columns
# columns_order = [
#     'Product', 'Variant', 'Price', 'Handle', 
#     'Inventory Management', 'Variant ID', 'Availability', 'SKU', 
#     'Product URL', 'Scraped at',
#     'Description', 'Hollistic Description'
# ]
# Save to CSV (index=False prevents that extra "0, 1, 2" column at the start)
df_merged.to_csv(r'C:\Users\moham\OneDrive\Desktop\WebScraping\legacy_app\output\baby_banana\baby_banana_compact.csv', index=False)
df_merged

,Product,Description,Price,Variant ID,SKU,Inventory Management,Handle,Availability,Product URL,Scraped at,Hollistic Description
0,Baby Banana Wet Bag,Carry all your essentials in this light weight...,150.0,44352175767724,nan,shopify,baby-banana-wet-bag,False,https://babybananaeg.com/products/baby-banana-...,2026/02/22 02:16:40 PM,Baby Banana Wet Bag: Carry all your essentials...
1,Baby fringes top,Handmade with love by Egyptian women especiall...,1000.0,"44287113527468, 44287092326572, 44287092359340...",nan,shopify,baby-fringes-top,True,https://babybananaeg.com/products/baby-fringes...,2026/02/22 02:16:34 PM,Baby fringes top: Handmade with love by Egypti...
2,Banana Tropicana,A fresh take on our best seller set that made ...,500.0,"41645215056044, 41645215088812, 41645215121580...",nan,shopify,banana-tropicana,False,https://babybananaeg.com/products/banana-tropi...,2026/02/22 02:16:30 PM,Banana Tropicana: A fresh take on our best sel...
3,Bandana Banana,A tribute to the cool kids on the block in a 9...,500.0,"41585045799084, 41585045831852, 41585045864620...",nan,shopify,bandana-banana,False,https://babybananaeg.com/products/bandana-banana,2026/02/22 02:16:41 PM,Bandana Banana: A tribute to the cool kids on ...
4,Beach Bum Set,Our first crop pants set comes in a breezy lin...,500.0,"41127727628460, 41127727661228, 41127727693996",nan,shopify,beach-bum-set,True,https://babybananaeg.com/products/beach-bum-set,2026/02/22 02:16:42 PM,Beach Bum Set: Our first crop pants set comes ...
5,Boho Crop Top,The Top that goes with anything and everything...,360.0,"41618376491180, 41618376523948, 41618360926380...",2.0,None,boho-crop-top,True,https://babybananaeg.com/products/boho-crop-top,2026/02/22 02:16:36 PM,Boho Crop Top: The Top that goes with anything...
6,Bubble Gum Bum White,Also available in black,500.0,"41127688929452, 41127688962220, 41127688994988...",nan,shopify,bubble-gum-bum-white,True,https://babybananaeg.com/products/bubble-gum-b...,2026/02/22 02:16:31 PM,Bubble Gum Bum White: Also available in black
7,California Girl,We paired our signature ruffled shorts with an...,900.0,"44349099770028, 44349099802796, 44349099835564...",nan,shopify,green-glam-copy,True,https://babybananaeg.com/products/green-glam-copy,2026/02/22 02:16:38 PM,California Girl: We paired our signature ruffl...
8,Cotton Candy Coverup Skirts,"the coverup you wont take off all summer, easy...",300.0,"42601610477740, 42601610510508, 42601613394092...",nan,shopify,copy-of-go-bananas-coverup,False,https://babybananaeg.com/products/copy-of-go-b...,2026/02/22 02:16:38 PM,Cotton Candy Coverup Skirts: the coverup you w...
9,Cotton candy fringe top,Handmade with love by Egyptian women especiall...,1000.0,"44287128305836, 44287075549356, 44287075582124...",nan,shopify,cotton-candy-fringe-top,True,https://babybananaeg.com/products/cotton-candy...,2026/02/22 02:16:33 PM,Cotton candy fringe top: Handmade with love by...


In [11]:
'''
Products Table
'''
df_products = df_merged
df_products = df_products.drop(columns=['Variant ID', 'Availability'])

# Split by comma and strip whitespace from each element
# 1. First, split the strings into lists (if you haven't already)
df_products['SKUs'] = df_products['SKU'].str.split(',')

# 2. Grab the first element of each list and strip whitespace
df_products['SKU'] = df_products['SKUs'].str[0].str.strip()
# 1. Clean up the decimals and whitespace within the lists first
# This removes '.0' and extra spaces from every item in every list
df_products['SKUs'] = df_products['SKUs'].apply(lambda x: [i.replace('.0', '').strip() for i in x])

# 2. Join the list elements into a single string separated by a space
df_products['SKUs'] = df_products['SKUs'].str.join(' ')

df_products.to_csv(r'C:\Users\moham\OneDrive\Desktop\WebScraping\legacy_app\output\baby_banana\baby_banana_products.csv', index = False)